# STARK-Amazon Data Exploration

Loads the QA dataset and Semi-structured Knowledge Base (SKB), inspects the test splits,
and characterises individual queries and product nodes before any pipeline code is written.

## 1. Load QA dataset and SKB

First run downloads to `~/.cache/huggingface/` — subsequent runs load from cache.

In [ ]:
from stark_qa import load_qa, load_skb

qa_dataset = load_qa('amazon')
skb = load_skb('amazon')

print(f'QA dataset size: {len(qa_dataset)}')
print(f'SKB size: {len(skb)}')

/Users/shreeshangaavin/ShreeShangaavi/master thesis/.venv/lib/python3.12/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Use file from /Users/shreeshangaavin/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/qa/amazon/stark_qa/stark_qa_human_generated_eval.csv.
Loading from /Users/shreeshangaavin/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/skb/amazon/processed!
Loading cached graph with meta link types ['brand', 'category', 'color']


In [1]:
from stark_qa import load_qa
qa_dataset = load_qa('amazon')
idx_split = qa_dataset.get_idx_split()
print('test size:', len(idx_split['test']))
print('test-0.1 size:', len(idx_split['test-0.1']))

/Users/shreeshangaavin/ShreeShangaavi/master thesis/.venv/lib/python3.12/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Use file from /Users/shreeshangaavin/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/qa/amazon/stark_qa/stark_qa_human_generated_eval.csv.
test size: 1642
test-0.1 size: 164


## 2. Inspect the official splits

In [ ]:
idx_split = qa_dataset.get_idx_split()
print('Available splits:', list(idx_split.keys()))

for split_name, ids in idx_split.items():
    print(f'  {split_name:12s}: {len(ids):,} queries')

In [ ]:
test_ids    = idx_split['test']      # full test set  (~1,638 queries) — used for Step B metrics
test_01_ids = idx_split['test-0.1']  # 10% split      (~164 queries)   — used for Step C and ablations

print(f'test     : {len(test_ids):,} queries')
print(f'test-0.1 : {len(test_01_ids):,} queries')

# Verify test-0.1 is a true subset of test
assert set(test_01_ids).issubset(set(test_ids)), 'test-0.1 is NOT a subset of test!'
print('Confirmed: test-0.1 is a subset of test')

## 3. Inspect a single query

In [ ]:
idx = test_ids[0]
query, answer_ids, _ = qa_dataset[idx]

print(f'Query:\n  {query}')
print(f'Answer node IDs ({len(answer_ids)}): {answer_ids[:5]}{'...' if len(answer_ids) > 5 else ''}')

In [ ]:
# Look at a few more queries to get a feel for the distribution
for i in range(5):
    q, ans, _ = qa_dataset[test_ids[i]]
    print(f'[{i}] answers={len(ans):2d} | {q[:100]}')

## 4. Inspect a product node (candidate card)

In [ ]:
node_id = answer_ids[0]
node = skb.get_node_by_id(node_id)
print(f'Node ID: {node_id}')
print(node)

In [ ]:
# What attributes / fields does a node have?
if hasattr(node, '__dict__'):
    print(list(vars(node).keys()))
elif isinstance(node, dict):
    print(list(node.keys()))

## 5. Inspect graph edges (relations used in Step A)

The five relation types Step A decomposes into:
`has_brand`, `has_category`, `has_color`, `also_bought`, `also_viewed`

In [ ]:
# What relation types exist in the SKB?
if hasattr(skb, 'relation_types'):
    print(skb.relation_types)
elif hasattr(skb, 'edge_types'):
    print(skb.edge_types)
else:
    # Fallback: inspect the skb object
    print(type(skb))
    print([attr for attr in dir(skb) if not attr.startswith('_')])

In [ ]:
# Sample edges for the first answer node
target_relations = ['has_brand', 'has_category', 'has_color', 'also_bought', 'also_viewed']

for rel in target_relations:
    try:
        neighbors = skb.get_neighbor_nodes(node_id, rel)
        print(f'{rel:15s}: {neighbors[:5]}')
    except Exception as e:
        print(f'{rel:15s}: ERROR — {e}')

## 6. Quick corpus statistics

In [ ]:
import numpy as np

# Distribution of answer set sizes across the test split
answer_sizes = [len(qa_dataset[i][1]) for i in test_ids]
print(f'Answer set size — min: {min(answer_sizes)}, max: {max(answer_sizes)}, '
      f'mean: {np.mean(answer_sizes):.1f}, median: {np.median(answer_sizes):.0f}')

In [ ]:
# Query length distribution (word count)
query_lengths = [len(qa_dataset[i][0].split()) for i in test_ids]
print(f'Query length (words) — min: {min(query_lengths)}, max: {max(query_lengths)}, '
      f'mean: {np.mean(query_lengths):.1f}')

## 7. Verify the text corpus used for BM25 (Step B)

BM25 needs a text representation of each node. The SKB provides this via `get_doc_info`.

In [ ]:
# Check what text retrieval methods the SKB exposes
print([attr for attr in dir(skb) if 'doc' in attr.lower() or 'text' in attr.lower() or 'corpus' in attr.lower()])

In [ ]:
# Get the text document for the first answer node — this is what BM25 indexes
try:
    doc = skb.get_doc_info(node_id, add_rel=True, compact=True)
    print(doc[:1000])
except Exception as e:
    print(f'get_doc_info failed: {e}')
    # Try alternate method names
    for method in ['get_doc', 'get_text', 'node_to_text']:
        if hasattr(skb, method):
            print(f'Found method: {method}')